# 06a. Reference data quality: PV circuits with constant reactive power

Some PV circuits in the monitoring data report reactive power that **never changes**: every interval in the fitted window has exactly the same Q (in practice, exactly 0). A real inverter's Q always moves a little from interval to interval, even with no Volt-VAr response, so a perfectly constant Q means the circuit is not measuring reactive power (e.g. the reactive-energy register is not counting). For those site-months the PV-circuit reference cannot tell us what the inverter did.

**The flag uses the PV-circuit data only** (never the net meter or the unmonitored verdict), so it is applied the same way whatever the unmonitored method concluded. Flagged site-months are treated as *reference not assessable*: their reference verdict is set to missing, not changed.

In [ ]:
import sys
from pathlib import Path

# Put the repo root on sys.path (same as the other notebooks).
_current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import duckdb
import numpy as np
import pandas as pd

from bms_sa_review.ami_analysis.lib import ami_fleet_vvar as Fleet
from bms_sa_review.ami_analysis.lib import ami_vvar_validation as Valid

## 1. Settings

In [ ]:
# Must match notebook 06 (main scenario)
MAIN = dict(v_lo=242, v_hi=253, p_min_frac=Fleet.P_MIN_FRAC, night_hours=(0, 4))
MIN_N_WINDOW = 20           # min. intervals in the fitted window
MIN_V_SPREAD = 5.0          # min. voltage range (V) of those intervals
POSITIVE_IS_FAIL = False    # positive class = conformant

# Flag level:
#   "site_month" -> flag each reference site-month whose Q is constant
#   "circuit"    -> flag every assessable month of a PV circuit whose Q is constant in >= CIRCUIT_SHARE of them
FLAG_LEVEL = "site_month"
CIRCUIT_SHARE = 0.5

TRUTH_FLAT = 0.5            # |reference slope| below this = "no response" (same as notebook 06, cell 27)

RESULTS_DIR = Fleet.store_dir() / "vvar_validation" / "all_sites" / f"band_{MAIN['v_lo']}_{MAIN['v_hi']}"
THREE_WAY = RESULTS_DIR / "three_way_site_month.csv"           # written by notebook 06
FLAGS_OUT = RESULTS_DIR / "reference_constant_q_flags.csv"
print(f"Results folder: {RESULTS_DIR}")

## 2. Load the reference fits and the three-way comparison

Reference (PV-circuit) fits are read from disk, as written by notebook 06. The cohort is taken from
`three_way_site_month.csv`, so it is exactly the 06 cohort.

In [ ]:
# Three-way comparison from notebook 06 (CSV stores booleans as text)
tw = pd.read_csv(THREE_WAY)
for c in ["pass_blind", "pass_truth", "pass_sa"]:
    tw[c] = tw[c].map({True: True, False: False, "True": True, "False": False}).astype("boolean")
cohort_sites = tw.site_id.unique()

# Reference fits, cohort only, with fit status
ref = Fleet.load_results(RESULTS_DIR / "ground_truth")
ref = ref.loc[ref.site_id.isin(cohort_sites)]
ref = Fleet.add_fit_status(ref, min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD)
ref_ok = ref.loc[ref.fit_status == "ok"].copy()
print(f"{len(cohort_sites):,} cohort sites; {len(ref_ok):,} assessable reference site-months")

## 3. The flag (PV-circuit data only)

In [ ]:
# Constant Q in the fitted window: slope exactly 0 AND no residual scatter at all
ref_ok["constant_q"] = (ref_ok.slope == 0) & (ref_ok.slope_stderr == 0)

flagged = ref_ok.loc[ref_ok.constant_q]
print(f"Constant-Q reference site-months: {len(flagged):,} of {len(ref_ok):,} ({100 * len(flagged) / len(ref_ok):.2f}%)")
print(f"  at {flagged.site_id.nunique():,} sites / {flagged.circuit_id.nunique():,} PV circuits")

# Sanity checks: R^2 should be exactly 1, and the constant value itself (= intercept, % of rating)
print(f"  R^2 == 1 for {(flagged.r_squared == 1).mean():.1%} of them")
display(flagged.intercept.describe().round(4).to_frame("constant Q (% of rating)"))

### 3.1 Month-level or circuit-level?

If a circuit's Q is constant in some months but not others, a month-level flag is enough. If circuits are constant in
(nearly) all their months, the two rules coincide and the problem is a circuit/device issue.

In [ ]:
per_circuit = (ref_ok.groupby("circuit_id")
                     .agg(site_id=("site_id", "first"), months_ok=("constant_q", "size"),
                          months_constant=("constant_q", "sum")))
per_circuit["share_constant"] = per_circuit.months_constant / per_circuit.months_ok
affected = per_circuit.loc[per_circuit.months_constant > 0]

print(f"{len(affected):,} PV circuits have at least one constant-Q month")
display(affected.share_constant.describe().round(3).to_frame("share of assessable months constant"))
print(f"Circuits with constant Q in >= {CIRCUIT_SHARE:.0%} of their months: "
      f"{(affected.share_constant >= CIRCUIT_SHARE).sum():,}")

flag_circuits = set(per_circuit.index[per_circuit.share_constant >= CIRCUIT_SHARE])
ref_ok["flag_circuit"] = ref_ok.circuit_id.isin(flag_circuits)
ref_ok["ref_flag"] = ref_ok.constant_q if FLAG_LEVEL == "site_month" else ref_ok.flag_circuit
print(f"\nFLAG_LEVEL = {FLAG_LEVEL!r}: {ref_ok.ref_flag.sum():,} reference site-months flagged "
      f"(site-month rule {ref_ok.constant_q.sum():,}; circuit rule {ref_ok.flag_circuit.sum():,})")

### 3.2 Who is affected (device type, OEM, install year)

In [ ]:
circ_meta = duckdb.sql(f"""
    SELECT circuit_id, device_type
    FROM read_parquet('{Fleet.Config.store_path("ami_circuit_metadata").as_posix()}')
""").df()
site_meta = duckdb.sql(f"""
    SELECT site_id, manufacturer, pv_install_date
    FROM read_parquet('{Fleet.Config.store_path("ami_site_metadata_residential").as_posix()}')
""").df()

d = (ref_ok.merge(circ_meta, on="circuit_id", how="left")
           .merge(site_meta, on="site_id", how="left")
           .assign(install_year=lambda x: pd.to_datetime(x.pv_install_date).dt.year))

for col in ["device_type", "manufacturer", "install_year"]:
    t = pd.DataFrame({
        "flagged_%": 100 * d.loc[d.ref_flag, col].value_counts(normalize=True),
        "all_assessable_%": 100 * d[col].value_counts(normalize=True),
        "flagged_n": d.loc[d.ref_flag, col].value_counts(),
    }).fillna(0).sort_values("flagged_n", ascending=False).head(8)
    display(t.round(1))

## 4. The flag does not select on the outcome

Cross-tab of flagged vs not-flagged site-months against the **unmonitored** verdict, on the site-months both methods
assessed. If the flag were picking false positives, it would only catch unmonitored passes; it also removes site-months
where both methods say non-conformant.

In [ ]:
keys = ["site_id", "year", "month"]
tw = tw.merge(ref_ok[keys + ["ref_flag"]], on=keys, how="left")
tw["ref_flag"] = tw.ref_flag.fillna(False).astype(bool)

both = tw.dropna(subset=["pass_blind", "pass_truth"])
display(pd.crosstab(both.ref_flag, both.pass_blind.astype(bool),
                    rownames=["reference flagged"], colnames=["unmonitored conformant"], margins=True))

## 5. Agreement with and without the flagged reference site-months

Flagged site-months keep their unmonitored and timestamp-method verdicts; only the reference verdict is set to missing.
This affects both pairs that use the reference.

In [ ]:
tw_clean = tw.copy()
tw_clean.loc[tw_clean.ref_flag, ["pass_truth", "slope_truth"]] = pd.NA

rows = []
for label, frame in [("all reference site-months", tw), ("constant-Q reference excluded", tw_clean)]:
    for common_only in (False, True):
        a = Valid.agreement_table(frame, positive_is_fail=POSITIVE_IS_FAIL, common_only=common_only)
        rows.append(a.assign(scenario=label, overlap="all three" if common_only else "own pair"))
agreement = pd.concat(rows, ignore_index=True)
display(agreement.set_index(["overlap", "tested", "reference", "scenario"])
        [["n", "pass_rate_tested_%", "pass_rate_reference_%", "TP", "FP", "FN", "TN",
          "accuracy", "precision", "recall", "specificity", "f1"]].sort_index().round(3))

## 6. What remains of the flat-reference cluster

The cluster in notebook 06 (cell 27) is "reference flat (|slope| < `TRUTH_FLAT`) and unmonitored passes". After the
flag, the part that remains has a near-zero but **non-constant** reference slope; these stay as genuine disagreements.

In [ ]:
b = tw.dropna(subset=["pass_blind", "pass_truth"])
cluster = b.loc[(b.slope_truth.abs() < TRUTH_FLAT) & b.pass_blind.astype(bool)]
remaining = cluster.loc[~cluster.ref_flag]
print(f"Cluster: {len(cluster):,} site-months ({cluster.site_id.nunique():,} sites)")
print(f"  removed by the flag: {cluster.ref_flag.sum():,}")
print(f"  remaining:           {len(remaining):,} site-months ({remaining.site_id.nunique():,} sites)")
display(remaining[["slope_blind", "r_squared_blind", "slope_truth", "r_squared_truth"]].describe().round(3))

## 7. Save the flags

In [ ]:
out = ref_ok[["site_id", "circuit_id", "year", "month", "constant_q", "flag_circuit", "ref_flag"]].copy()
out["flag_level"] = FLAG_LEVEL
out.to_csv(FLAGS_OUT, index=False)
agreement.to_csv(RESULTS_DIR / "agreement_constant_q_excluded.csv", index=False)
print(f"Saved {len(out):,} rows ({out.ref_flag.sum():,} flagged) -> {FLAGS_OUT}")